Cell 1 – Imports

In [41]:
# Import os to set environment variables before TensorFlow is loaded
import os

# Disable GPU so TensorFlow will run only on CPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Print confirmation
print("Step 1 completed: GPU disabled. TensorFlow will use CPU only.")

Step 1 completed: GPU disabled. TensorFlow will use CPU only.


In [42]:
!pip install tensorflow


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [43]:
# Import the 'os' module to work with file paths and check if files exist
import os

# Import the EfficientNetB7 model and its preprocessing function
from tensorflow.keras.applications.efficientnet import EfficientNetB7, preprocess_input

# Import functions to load an image and convert it into a NumPy array
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Import the Model class to create a custom feature extractor model
from tensorflow.keras.models import Model

# Import pickle to save the extracted feature vector into a file
import pickle

# Print a message to confirm imports are complete
print("Step 1 completed: All libraries imported successfully.")

Step 1 completed: All libraries imported successfully.


Cell 2 – Load the EfficientNet-B7 model

In [44]:
# Print a message before loading the model
print("Loading EfficientNetB7 model with ImageNet weights...")

# Load the EfficientNetB7 model with pre-trained ImageNet weights
base_model = EfficientNetB7(weights='imagenet')

# Print a message after the model is loaded
print("Step 2 completed: EfficientNetB7 model loaded successfully.")

Loading EfficientNetB7 model with ImageNet weights...
Step 2 completed: EfficientNetB7 model loaded successfully.


Cell 3 – Restructure the model for feature extraction

In [45]:
# Print a message before creating the feature extractor model
print("Creating feature extractor model...")

# Create a new model using the same input as EfficientNetB7
# Use the second-to-last layer as output to get feature vectors
feature_extractor_model = Model(
    inputs=base_model.inputs,            # Use the original model input
    outputs=base_model.layers[-2].output # Use the layer before final classification
)

# Print a message after creating the model
print("Step 3 completed: Feature extractor model created successfully.")

Creating feature extractor model...
Step 3 completed: Feature extractor model created successfully.


Cell 4 – Show model summary

In [46]:
# Print a message before showing the model summary
print("Displaying feature extractor model summary...")

# Print the summary of the feature extractor model
feature_extractor_model.summary()

# Print a message after the summary is displayed
print("Step 4 completed: Model summary displayed.")

Displaying feature extractor model summary...


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 600, 600,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_8         │ (None, 600, 600,  │          0 │ input_layer_4[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_4     │ (None, 600, 600,  │          7 │ rescaling_8[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_9         │ (None, 600, 600,  │          0 │ normalization_4[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 601, 601,  │          0 │ rescaling_9[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 300, 300,  │      1,728 │ stem_conv_pad[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 300, 300,  │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 300, 300,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 300, 300,  │        576 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 300, 300,  │        256 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 300, 300,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 64)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 64)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 16)  │      1,040 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 64)  │      1,088 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 300, 300,  │          0 │ block1a_activati… │
│ (Multiply)          │ 64)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 300, 300,  │      2,048 │ block1a_se_excit

 Total params: 64,097,687 (244.51 MB)

 Trainable params: 63,786,960 (243.33 MB)

 Non-trainable params: 310,727 (1.19 MB)

Step 4 completed: Model summary displayed.


Cell 5 – Define the image path and output file path

In [47]:
# Define the input image file name
image_path = 'front_mask.png'

# Define the output pickle file name
features_pickle_path = 'front_mask_feature.pkl'

# Print the selected image path
print(f"Input image path: {image_path}")

# Print the output pickle path
print(f"Output feature file: {features_pickle_path}")

# Check whether the image file exists
if os.path.exists(image_path):
    print("Step 5 completed: Image file found.")
else:
    print("Step 5 error: Image file not found. Please check the file path.")

Input image path: front_mask.png
Output feature file: front_mask_feature.pkl
Step 5 completed: Image file found.


Cell 6 – Load and preprocess the image

In [48]:
# Print a message before loading the image
print("Loading and preprocessing the image...")

# Load the image and resize it to 600x600 as required by EfficientNetB7
img = load_img(image_path, target_size=(600, 600))

# Print the original loaded image information
print("Image loaded successfully.")

# Convert the PIL image to a NumPy array
img = img_to_array(img)

# Print the shape after conversion
print(f"Image shape after conversion to array: {img.shape}")

# Add a batch dimension to make the shape (1, height, width, channels)
img = img.reshape(1, img.shape[0], img.shape[1], img.shape[2])

# Print the shape after adding batch dimension
print(f"Image shape after adding batch dimension: {img.shape}")

# Preprocess the image using EfficientNet preprocessing
img = preprocess_input(img)

# Print a message after preprocessing
print("Step 6 completed: Image preprocessed successfully.")

Loading and preprocessing the image...
Image loaded successfully.
Image shape after conversion to array: (600, 600, 3)
Image shape after adding batch dimension: (1, 600, 600, 3)
Step 6 completed: Image preprocessed successfully.


Cell 7 – Extract features from the image

In [49]:
# Print a message before feature extraction
print("Extracting features from the image...")

# Pass the preprocessed image into the feature extractor model
feature_vector = feature_extractor_model.predict(img, verbose=1)

# Print the shape of the extracted feature vector
print(f"Feature extraction completed.")

# Print the feature vector shape
print(f"Feature vector shape: {feature_vector.shape}")

# Print a small preview of the feature vector
print("First 10 feature values:")
print(feature_vector[0][:10])

# Confirm step completion
print("Step 7 completed: Features extracted successfully.")

Extracting features from the image...
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Feature extraction completed.
Feature vector shape: (1, 2560)
First 10 feature values:
[-0.0537962  -0.08206413 -0.16709389  0.60237557  0.29315498 -0.11175759
 -0.17305797 -0.18972133 -0.07434149  0.21857037]
Step 7 completed: Features extracted successfully.


Cell 8 – Save the extracted features as a pickle file

In [50]:
# Print a message before saving the feature vector
print("Saving the extracted feature vector to a pickle file...")

# Open the output file in binary write mode
with open(features_pickle_path, 'wb') as f:
    # Save the feature vector into the pickle file
    pickle.dump(feature_vector, f)

# Print a message after saving
print(f"Step 8 completed: Feature vector saved to '{features_pickle_path}'.")

Saving the extracted feature vector to a pickle file...
Step 8 completed: Feature vector saved to 'front_mask_feature.pkl'.
